# I-03. AI 코딩 도우미로 데이터 분석하기

「지구과학 데이터 사이언스」 책의 코드입니다. 설명은 책을 참고하세요.

**준비** (책에는 싣지 않은 공통 셀)

In [ ]:
# 필요한 모듈 정의 (버전 명시 가능)
packages_str = "numpy, pandas, matplotlib, version_information"

In [ ]:
# 공통 함수(dses_utils) 불러오기 — 코랩/로컬 모두 동작(별도 설치 불필요)
import os, sys, importlib, urllib.request

_url = "https://raw.githubusercontent.com/Kiehyun/dses_utils/main/dses_utils.py"
try:
    _new = urllib.request.urlopen(_url, timeout=10).read()  # 최신 내용을 메모리로 받아옴
    _old = open("dses_utils.py", "rb").read() if os.path.exists("dses_utils.py") else None
    if _new != _old:  # 내용이 다를 때만 덮어씀
        with open("dses_utils.py", "wb") as _f:
            _f.write(_new)
        if "dses_utils" in sys.modules:                       # 이미 불러왔다면 다시 불러옴
            importlib.reload(sys.modules["dses_utils"])
except Exception:
    pass  # 인터넷이 안 되면 기존에 받아둔 dses_utils.py 사용

import dses_utils as du
du.check_and_install(packages_str)  # 패키지 확인·설치와 버전 출력
du.set_korean_font()                # 그래프의 한글 글꼴 설정
proj_dir = du.make_project_dir(du.current_notebook_name())  # 데이터 저장 폴더 생성

**code Ⅰ-12.**

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)                       # 같은 난수가 나오도록 씨앗 고정
days = pd.date_range("2025-01-01", "2025-12-31", freq="D")
doy = days.dayofyear.to_numpy()

season = -np.cos(2 * np.pi * (doy - 15) / 365)  # 1월 중순 -1, 7월 중순 +1
tmean = 12.5 + 14.5 * season                    # 일평균기온의 계절 변화(°C)
# 일교차(반년 주기: 4·10월 최대, 1·7월 최소)
drange = 8.5 + 1.5 * np.cos(4 * np.pi * (doy - 105) / 365)

df = pd.DataFrame({
    "tmax": tmean + drange / 2 + rng.normal(0, 2.0, len(days)),
    "tmin": tmean - drange / 2 + rng.normal(0, 2.0, len(days)),
}, index=days)
df.head()

**code Ⅰ-13.**

In [ ]:
import matplotlib.pyplot as plt

ai = df.copy()
ai["month"] = ai.index.month.astype(str) + "월"           # 1월, 2월, … 을 글자로 만든다
ai["range"] = ai["tmax"] - ai["tmin"]
monthly = ai.groupby("month")[["tmax", "tmin", "range"]].mean()

fig, ax = plt.subplots(figsize=(8.5, 4))
monthly.plot(ax=ax, marker="o")
ax.set_xlabel("월")
ax.set_ylabel("기온 (°C)")
ax.set_title("AI가 처음 그린 그래프")
plt.show()
print(list(monthly.index))

**code Ⅰ-14.**

In [ ]:
expected = [f"{m}월" for m in range(1, 13)]
print("월 순서가 1월~12월인가? ", list(monthly.index) == expected)

july_direct = df.loc[df.index.month == 7, "tmax"].mean()     # 다른 방법으로 다시 계산
print("7월 평균 최고기온이 같은가?",
      np.isclose(july_direct, monthly.loc["7월", "tmax"]))

**code Ⅰ-15.**

In [ ]:
import os

fixed = (df.assign(range=df["tmax"] - df["tmin"])
         .groupby(df.index.month)[["tmax", "tmin", "range"]].mean())

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(fixed.index, fixed["tmax"], marker="o", label="평균 최고기온")
ax.plot(fixed.index, fixed["tmin"], marker="o", label="평균 최저기온")
ax.plot(fixed.index, fixed["range"], marker="s", label="평균 일교차")
ax.set_xticks(range(1, 13))
ax.set_xticklabels([f"{m}월" for m in range(1, 13)])
ax.set_ylabel("기온 (°C)")
ax.set_title("고친 그래프 — 월별 기온과 일교차")
ax.grid(True, alpha=0.4)
ax.legend()
du.save_figure(fig, os.path.join(str(proj_dir), f"{proj_dir.name}_01"))
plt.show()

print("월 순서가 1~12인가?", list(fixed.index) == list(range(1, 13)))